In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

import celltypist
from celltypist import models
from celltypist.models import Model

# preprocessing

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_01.h5ad")
adata_scraw=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")

In [ ]:
# preprocessing pipeline
sc.pp.normalize_total(adata_scraw)
sc.pp.log1p(adata_scraw)
adata_scraw.raw = adata_scraw.copy()

sc.pp.highly_variable_genes(adata_scraw)
#adata_scraw = adata_scraw[:, adata_scraw.var['highly_variable']]

sc.pp.scale(adata_scraw, max_value=10)
sc.tl.pca(adata_scraw, svd_solver='arpack')

In [ ]:
adata_scraw

In [ ]:
# t-sne 
sc.tl.tsne(adata_scraw, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_scraw, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_scraw)
sc.pl.umap(adata_scraw, color=['sample'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_scraw.obsm['X_pca'],adata_scraw.obs, 'sample')

# Harmony 결과를 저장
adata_scraw.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_scraw, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_scraw)
sc.pl.umap(adata_scraw, color=['state'])

In [ ]:
sc.tl.rank_genes_groups(
    adata_scraw,
    groupby='state',
    method='wilcoxon',
    use_raw=True,
    pts=True,
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_scraw, n_genes=10)

In [ ]:
rank_df_state = sc.get.rank_genes_groups_df(adata_scraw, group=None)
top_genes_state = rank_df_state.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scraw,
    var_names=top_genes_state,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
deg_state = sc.get.rank_genes_groups_df(adata_scraw, group=None)
deg_state 

In [ ]:
deg_state.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNAseq_MASLD_state_DEG.csv")

In [ ]:
adata_scraw.write("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNAseq_001.h5ad")

# celltypist

In [ ]:
sc.tl.leiden(adata_scraw, resolution=0.5, key_added="leiden")
sc.pl.umap(adata_scraw, color=['leiden'])

In [ ]:
sc.pl.umap(
    adata_scraw,
    color='leiden',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
predictions_cluster = celltypist.annotate(adata_scraw, model='Healthy_Human_Liver.pkl', majority_voting=True, over_clustering=adata_scraw.obs["leiden"]
)
adata_scraw =predictions_cluster.to_adata()

In [ ]:
sc.pl.umap(
    adata_scraw,
    color='majority_voting',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pl.umap(adata_scraw[adata_scraw.obs['state'] == 'healthy'], color='majority_voting', title='Healthy')
sc.pl.umap(adata_scraw[adata_scraw.obs['state'] == 'NAFLD'], color='majority_voting', title='MASLD')

In [ ]:
adata_scraw=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_monoMAC_detail.h5ad")
adata_scraw

In [ ]:
adata_scraw.obs['celltype'] = adata_scraw.obs['majority_voting']

In [ ]:
adata_scraw.obs['state'] = adata_scraw.obs['state'].replace({
    'healthy': 'Healthy',
    'NAFLD': 'MASLD'})

In [ ]:
adata_scraw.obs['state'].value_counts()

In [ ]:
adata_scraw.obs['celltype'] = adata_scraw.obs['celltype'].replace({
    'T cells': 'Lymphoid',
    'Resident NK': 'Lymphoid',
    'Mono+mono derived cells': 'Myeloid',
    'Endothelial cells': 'Endothelial',
    'Circulating NK/NKT': 'Lymphoid',
    'Macrophages': 'Myeloid',
    'Fibroblasts': 'Stellete',
    'Cholangiocytes': 'Cholangiocyte',
    'B cells': 'B cell',
    'cDC1s': 'Myeloid',
    'pDCs': 'pDC',
    'Basophils': 'Basophils',
    'Plasma cells': 'Plasma cells'
})

sc.pl.umap(adata_scraw, color=['celltype'])

In [ ]:
sc.tl.rank_genes_groups(
    adata_scraw,
    groupby='celltype',
    method='wilcoxon',
    use_raw=True,
    pts=True,
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_scraw, n_genes=10)

In [ ]:
rank_df_state = sc.get.rank_genes_groups_df(adata_scraw, group=None)
top_genes_state = rank_df_state.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scraw,
    var_names=top_genes_state,
    groupby="celltype",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
# 그룹별 샘플링 수
n_cells_per_group = 100
groups = adata_scraw.obs['celltype'].unique()

# 샘플링된 셀들의 인덱스 저장
selected_indices = []

for group in groups:
    group_cells = adata_scraw.obs[adata_scraw.obs['celltype'] == group].index
    n = min(len(group_cells), n_cells_per_group)  # 그룹 크기가 작을 경우 자동 조절
    sampled = np.random.choice(group_cells, n, replace=False)
    selected_indices.extend(sampled)

adata_sampled = adata_scraw[selected_indices]

sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=10,
    groupby="celltype",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=False
)

In [ ]:
deg_celltype = sc.get.rank_genes_groups_df(adata_scraw, group=None)
deg_celltype

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ── SLC7A5 발현값 추출 (obs와 index 정렬 보장) ───────────────────────────────
gene = "SLC7A5"

expr_df = sc.get.obs_df(adata_scraw, keys=[gene], use_raw=True)

df = pd.DataFrame({
    "celltype": adata_scraw.obs["celltype"].values,
    "state":    adata_scraw.obs["state"].values,
    "expr":     expr_df[gene].values
})

celltypes = sorted(df["celltype"].unique())

# ── 유의성 기호 ───────────────────────────────────────────────────────────────
def sig_label(p):
    if p < 0.001:   return "***"
    elif p < 0.01:  return "**"
    elif p < 0.05:  return "*"
    else:           return "ns"

# ── Boxplot ───────────────────────────────────────────────────────────────────
n_ct = len(celltypes)
fig, axes = plt.subplots(1, n_ct, figsize=(3 * n_ct, 5), sharey=False)
if n_ct == 1:
    axes = [axes]

state_order = ["Healthy", "MASLD"]
palette     = {"Healthy": "#4878CF", "MASLD": "#D65F5F"}

for ax, ct in zip(axes, celltypes):
    sub = df[df["celltype"] == ct]

    sns.boxplot(
        data=sub, x="state", y="expr",
        order=state_order, palette=palette,
        width=0.5, linewidth=0.8,
        fliersize=1.5, ax=ax
    )

    # Mann-Whitney U test (scRNA-seq는 비정규분포 → 비모수 검정)
    h_vals = sub.loc[sub["state"] == "Healthy", "expr"].values
    m_vals = sub.loc[sub["state"] == "MASLD",   "expr"].values

    if len(h_vals) > 0 and len(m_vals) > 0:
        _, pval = stats.mannwhitneyu(h_vals, m_vals, alternative="two-sided")
        label = sig_label(pval)
    else:
        pval, label = np.nan, "N/A"

    # bracket + 유의성 표시
    y_max  = sub["expr"].max()
    y_top  = y_max * 1.08
    y_text = y_max * 1.12
    ax.plot([0, 0, 1, 1], [y_top, y_top * 1.02, y_top * 1.02, y_top],
            lw=0.8, color="black")
    ax.text(0.5, y_text, f"{label}\np={pval:.2e}",
            ha="center", va="bottom", fontsize=7)

    ax.set_title(ct, fontsize=9, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("log-normalized expression" if ct == celltypes[0] else "")
    ax.tick_params(axis="x", labelsize=7)

fig.suptitle(f"{gene} expression per cell type (Healthy vs MASLD)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("SLC7A5_celltype_boxplot.pdf", bbox_inches="tight")
plt.show()

In [ ]:
n_ct = len(celltypes)
fig, axes = plt.subplots(1, n_ct, figsize=(3 * n_ct, 5), sharey=False)
if n_ct == 1:
    axes = [axes]

for ax, ct in zip(axes, celltypes):
    sub = df[df["celltype"] == ct]

    sns.violinplot(
        data=sub, x="state", y="expr",
        order=state_order, palette=palette,
        width=0.7, linewidth=0.8,
        inner="box", cut=0, ax=ax
    )

    # Mann-Whitney U test
    h_vals = sub.loc[sub["state"] == "Healthy", "expr"].values
    m_vals = sub.loc[sub["state"] == "MASLD",   "expr"].values

    if len(h_vals) > 0 and len(m_vals) > 0:
        _, pval = stats.mannwhitneyu(h_vals, m_vals, alternative="two-sided")
        label = sig_label(pval)
    else:
        pval, label = np.nan, "N/A"

    # bracket + 유의성 표시
    y_max  = sub["expr"].max()
    y_top  = y_max * 1.08
    y_text = y_max * 1.12
    ax.plot([0, 0, 1, 1], [y_top, y_top * 1.02, y_top * 1.02, y_top],
            lw=0.8, color="black")
    ax.text(0.5, y_text, f"{label}\np={pval:.2e}",
            ha="center", va="bottom", fontsize=7)

    ax.set_title(ct, fontsize=9, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("log-normalized expression" if ct == celltypes[0] else "")
    ax.tick_params(axis="x", labelsize=7)

fig.suptitle(f"{gene} expression per cell type (Healthy vs MASLD)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("SLC7A5_celltype_violinplot.pdf", bbox_inches="tight")
plt.show()

In [ ]:
  n_ct = len(celltypes)                                                                                                                     
  fig, axes = plt.subplots(1, n_ct, figsize=(3 * n_ct, 5), sharey=False)                                                                    
  if n_ct == 1:                                                                                                                             
      axes = [axes]
                                                                                                                                            
  for ax, ct in zip(axes, celltypes):                                                                                                       
      sub = df[df["celltype"] == ct]
                                                                                                                                            
      sns.violinplot(
          data=sub, x="state", y="expr",
          order=state_order, palette=palette,
          width=0.7, linewidth=0.8,
          inner="box", cut=0, ax=ax                                                                                                         
      )
                                                                                                                                            
      # Mann-Whitney U test
      h_vals = sub.loc[sub["state"] == "Healthy", "expr"].values
      m_vals = sub.loc[sub["state"] == "MASLD",   "expr"].values                                                                            
  
      if len(h_vals) > 0 and len(m_vals) > 0:                                                                                               
          _, pval = stats.mannwhitneyu(h_vals, m_vals, alternative="two-sided")
          label = sig_label(pval)                                                                                                           
                  
          # ── 어느 그룹이 더 높은지 판별 (중앙값 기준) ──────────────────────                                                              
          h_med, m_med = np.median(h_vals), np.median(m_vals)
          if m_med > h_med:                                                                                                                 
              higher_group = "MASLD"                                                                                                        
              direction_label = "MASLD ↑"
          else:                                                                                                                             
              higher_group = "Healthy"                                                                                                      
              direction_label = "Healthy ↑"
      else:                                                                                                                                 
          pval, label = np.nan, "N/A"
          higher_group = None
          direction_label = ""                                                                                                              
  
      # bracket + 유의성 표시                                                                                                               
      y_max  = sub["expr"].max()
      y_top  = y_max * 1.08                                                                                                                 
      y_text = y_max * 1.12
      ax.plot([0, 0, 1, 1], [y_top, y_top * 1.02, y_top * 1.02, y_top],                                                                     
              lw=0.8, color="black")                                                                                                        
      ax.text(0.5, y_text,                                                                                                                  
              f"{label}\np={pval:.2e}\n{direction_label}",   # ← 방향 추가                                                                  
              ha="center", va="bottom", fontsize=7,                                                                                         
              color=palette.get(higher_group, "black"))       # ← 높은 쪽 색상
                                                                                                                                            
      ax.set_title(ct, fontsize=9, fontweight="bold")                                                                                       
      ax.set_xlabel("")                                                                                                                     
      ax.set_ylabel("log-normalized expression" if ct == celltypes[0] else "")                                                              
                  
      # ── 더 높은 그룹의 x축 라벨 굵게 + 색상 강조 ─────────────────────────                                                               
      tick_labels = ax.set_xticklabels(state_order, fontsize=7)
      for tick, s in zip(tick_labels, state_order):                                                                                         
          if s == higher_group:
              tick.set_fontweight("bold")                                                                                                   
              tick.set_fontsize(8)
              tick.set_color(palette[s])                                                                                                    
  
  fig.suptitle(f"{gene} expression per cell type (Healthy vs MASLD)",                                                                       
               fontsize=12, y=1.02)
  plt.tight_layout()                                                                                                                        
  plt.savefig("SLC7A5_celltype_violinplot.pdf", bbox_inches="tight")
  plt.show()         

In [ ]:
  from scipy.stats import fisher_exact                                                                                                      
                                                                                                                                            
  n_ct = len(celltypes)
  fig, axes = plt.subplots(1, n_ct, figsize=(3 * n_ct, 5), sharey=False)                                                                    
  if n_ct == 1:   
      axes = [axes]
                                                                                                                                            
  for ax, ct in zip(axes, celltypes):
      sub = df[df["celltype"] == ct]                                                                                                        
                  
      h_vals = sub.loc[sub["state"] == "Healthy", "expr"].values
      m_vals = sub.loc[sub["state"] == "MASLD",   "expr"].values
                                                                                                                                            
      # ── 평균 ± SEM 바 플랏 ────────────────────────────────────────────────                                                              
      means  = [np.mean(h_vals), np.mean(m_vals)]                                                                                           
      sems   = [stats.sem(h_vals), stats.sem(m_vals)]                                                                                       
      colors = [palette["Healthy"], palette["MASLD"]]                                                                                       
  
      ax.bar(                                                                                                                               
          state_order, means, yerr=sems,
          color=colors, width=0.5, capsize=4,
          linewidth=0.8, edgecolor="black", error_kw={"lw": 0.8}                                                                            
      )
                                                                                                                                            
      # ── Fisher's exact test (발현 여부 이분화, MASLD > Healthy 단측) ──────                                                              
      if len(h_vals) > 0 and len(m_vals) > 0:
          # 2×2 contingency table                                                                                                           
          # rows: MASLD / Healthy                                                                                                           
          # cols: expressed(>0) / not-expressed(=0)
          table = [                                                                                                                         
              [np.sum(m_vals > 0), np.sum(m_vals == 0)],
              [np.sum(h_vals > 0), np.sum(h_vals == 0)],                                                                                    
          ]                                                                                                                                 
          _, pval = fisher_exact(table, alternative="greater")  # MASLD 발현 비율이 더 높다
          label = sig_label(pval)                                                                                                           
      else:       
          pval, label = np.nan, "N/A"                                                                                                       
                  
      # ── bracket + 유의성 표시 ─────────────────────────────────────────────                                                              
      y_max  = max(m + s for m, s in zip(means, sems))
      y_top  = y_max * 1.12                                                                                                                 
      y_text = y_max * 1.18
      ax.plot([0, 0, 1, 1], [y_top, y_top * 1.02, y_top * 1.02, y_top],                                                                     
              lw=0.8, color="black")                                                                                                        
      ax.text(0.5, y_text,
              f"{label}\np={pval:.2e}\n(MASLD ↑)",                                                                                          
              ha="center", va="bottom", fontsize=7,                                                                                         
              color=palette["MASLD"])
                                                                                                                                            
      ax.set_title(ct, fontsize=9, fontweight="bold")                                                                                       
      ax.set_xlabel("")
      ax.set_ylabel("Mean log-normalized expression" if ct == celltypes[0] else "")                                                         
      ax.tick_params(axis="x", labelsize=7)

  fig.suptitle(                                                                                                                             
      f"{gene} expression per cell type (Healthy vs MASLD)\n"
      "Fisher's exact test  ·  H₁: MASLD expressing fraction > Healthy",                                                                    
      fontsize=11, y=1.03                                                                                                                   
  )
  plt.tight_layout()                                                                                                                        
  plt.savefig("SLC7A5_celltype_barplot.pdf", bbox_inches="tight")
  plt.show()

In [ ]:
# 2. crosstab 생성
cluster_labels = adata_scraw.obs['celltype']
batch_labels = adata_scraw.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_celltypist_IPF_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
adata_scraw.obs['sample'].value_counts()

In [ ]:
import re           
def get_batch(sample):                      
      m = re.match(r'(Healthy\d+)', sample)
      return m.group(1) if m else sample      
                                                                                                                                            
adata_scraw.obs['batch'] = adata_scraw.obs['sample'].apply(get_batch)
                                                                                                                                            
print(adata_scraw.obs['batch'].value_counts())

In [ ]:
# 2. crosstab 생성
cluster_labels = adata_scraw.obs['celltype']
batch_labels = adata_scraw.obs['batch']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(10, 6),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_celltypist_IPF_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
adata_scraw.write("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNAseq_001.h5ad")

# Myeloid sub clustering analysis

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNAseq_001.h5ad")
adata_scraw=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")

In [ ]:
print(adata_scrna)
print(adata_scraw)

In [ ]:
adata_scraw.obs['celltype'] = adata_scrna.obs['celltype']

In [ ]:
adata_myl = adata_scraw[adata_scraw.obs['celltype'].isin(['Myeloid'])].copy()
adata_myl

In [ ]:
# preprocessing pipeline
sc.pp.normalize_total(adata_myl)
sc.pp.log1p(adata_myl)
adata_myl.raw = adata_myl.copy()

sc.pp.highly_variable_genes(adata_myl)
#adata_scraw = adata_scraw[:, adata_scraw.var['highly_variable']]

sc.pp.scale(adata_myl, max_value=10)
sc.tl.pca(adata_myl, svd_solver='arpack')

In [ ]:
# t-sne 
sc.tl.tsne(adata_myl, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_myl, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_myl)
sc.pl.umap(adata_myl, color=['sample'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_myl.obsm['X_pca'],adata_myl.obs, 'sample')

# Harmony 결과를 저장
adata_myl.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_myl, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_myl)

In [ ]:
sc.pl.umap(adata_myl, color=['state'])

In [ ]:
sc.pl.umap(
    adata_myl,
    color='10_cluster',
    #legend_loc='on data',
    legend_fontsize=14,
    frameon=False,
    title='subclustering_10',
    size=10
)

In [ ]:
sc.tl.rank_genes_groups(
    adata_myl,
    groupby='10_cluster',
    method='wilcoxon',
    use_raw=True,
    pts=True,
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_myl, n_genes=10)

In [ ]:
rank_df_myl = sc.get.rank_genes_groups_df(adata_myl, group=None)
top_genes_myl = rank_df_myl.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_myl,
    var_names=top_genes_myl,
    groupby="10_cluster",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
deg_myl = sc.get.rank_genes_groups_df(adata_myl, group=None)
deg_myl

In [ ]:
# 그룹별 샘플링 수
n_cells_per_group = 100
groups = adata_myl.obs['10_cluster'].unique()

# 샘플링된 셀들의 인덱스 저장
selected_indices = []

for group in groups:
    group_cells = adata_myl.obs[adata_myl.obs['10_cluster'] == group].index
    n = min(len(group_cells), n_cells_per_group)  # 그룹 크기가 작을 경우 자동 조절
    sampled = np.random.choice(group_cells, n, replace=False)
    selected_indices.extend(sampled)

adata_sampled = adata_myl[selected_indices]

sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=10,
    groupby="10_cluster",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=False
)

In [ ]:
adata_myl.obs['state'] = adata_myl.obs['state'].replace({
    'healthy': 'Healthy',
    'NAFLD': 'MASLD'})

In [ ]:
# 2. crosstab 생성
cluster_labels = adata_myl.obs['10_cluster']
batch_labels = adata_myl.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_celltypist_IPF_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
adata_myl.write("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNAseq_myl_001.h5ad")

# myeloid annotation

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNAseq_myl_001.h5ad")
adata

In [ ]:
genes = ['CD14','FCGR3A']  # 예시 유전자 리스트
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata,
        keys=gene,
        groupby='10_cluster',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# FCER1A, and Mo-2 via increased expression of CSF3R, FCAR, and SELL
genes = ['FCER1A','CSF3R','FCAR','SELL']  # 예시 유전자 리스트
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata,
        keys=gene,
        groupby='10_cluster',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['FCN1', 'S100A8', 'S100A9', 'S100A12', 'VCAN', 'CSF3R', 'IL1RN', 'CXCL8', 'C5AR1', 'TREM1','FPR1','HIF1A']  # 예시 유전자 리스트
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata,
        keys=gene,
        groupby='10_cluster',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['TIMD4','CX3CR1','CD5L','VCAM1']  # 예시 유전자 리스트
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata,
        keys=gene,
        groupby='10_cluster',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
adata.obs['cellannot'] = adata.obs['10_cluster']

In [ ]:
adata.obs['cellannot'] = adata.obs['cellannot'].replace({
    '0': 'Classical monocyte',
    '1': 'aKC',
    '2': 'cDC2',
    '3': 'Patrolling monocyte',
    '4': 'LAM',
    '5': 'SLC7A5 high monocyte',
    '6': 'T/Nk doublet',
    '7': 'hoKC',
    '8': 'cDC1',
    '9': 'MigcDC'
})

In [ ]:
sc.pl.umap(
    adata,
    color='cellannot',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=15
)

In [ ]:
# 2. crosstab 생성
cluster_labels = adata.obs['cellannot']
batch_labels = adata.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_celltypist_IPF_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd
# 상태 및 클러스터 정보
state = adata.obs['state']
cluster = adata.obs['cellannot']

# 클러스터 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters):
    # 각 조건별 개수 계산
    donor_in = ((state == "Healthy") & (cluster == clus)).sum()
    donor_out = ((state == "Healthy") & (cluster != clus)).sum()
    ipf_in = ((state == "MASLD") & (cluster == clus)).sum()
    ipf_out = ((state == "MASLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [ipf_in, ipf_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out)
    ipf_frac = ipf_in / (ipf_in + ipf_out)
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "ipf_fraction": ipf_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 정렬
df_result = df_result.sort_values("cluster")
print(df_result)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(11, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='healthy')
plt.bar(x + width/2, df_result['ipf_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'], rotation=15, ha='right')
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['ipf_fraction'][i])
        plt.text(x[i], 0.34, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()